In [1]:
import xarray
from pyproj import Transformer
import numpy as np
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from tqdm import tqdm

import torch
from torch.utils.data import DataLoader, TensorDataset

import presto
from presto.dataops.dataset import (
    FranceCropsMiniDataset,
)
from pathlib import Path

# this is to silence the xarray deprecation warning.
# Our version of xarray is pinned, but we'll need to fix this
# when we upgrade
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

/home/p.kuznetsov/presto/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Using Presto as a feature extractor for a random forest

One way to use Presto is as a feature extractor for a simple model (e.g. a Random Forest). We do this below.

In [2]:
batch_size = 64

state_dict = torch.load("/home/p.kuznetsov/presto/output/2025_05_01_13_59_40_578691/models/0.pt")
pretrained_model = presto.Presto.construct(max_sequence_length=60)
pretrained_model.load_state_dict(state_dict=state_dict)
pretrained_model.eval()

Presto(
  (encoder): Encoder(
    (eo_patch_embed): ModuleDict(
      (S2_B1): Linear(in_features=1, out_features=128, bias=True)
      (S2_RGB): Linear(in_features=3, out_features=128, bias=True)
      (S2_Red_Edge): Linear(in_features=3, out_features=128, bias=True)
      (S2_NIR_10m): Linear(in_features=1, out_features=128, bias=True)
      (S2_NIR_20m): Linear(in_features=1, out_features=128, bias=True)
      (S2_NIR_60m): Linear(in_features=1, out_features=128, bias=True)
      (S2_SWIR): Linear(in_features=2, out_features=128, bias=True)
      (NDVI): Linear(in_features=1, out_features=128, bias=True)
    )
    (blocks): ModuleList(
      (0-1): 2 x Block(
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=128, out_features=384, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=128, out_

We will start by constructing features for the training data, and using this to train a Random Forest.

In [ ]:
train_dataset = FranceCropsMiniDataset(
    split="validation",
    directory=Path("/home/p.kuznetsov/presto/francecrops_mini"),
    mask_params=None,
    shuffle=False,
    seed=42,
    cache_dir="./cache_train_mini"
)

Saving the dataset (1/1 shards): 100%|█████████████████████████████████| 20000/20000 [00:00<00:00, 171484.10 examples/s]


: 

In [4]:
test_dataset = FranceCropsMiniDataset(
    split="test",
    directory=Path("/home/p.kuznetsov/presto/francecrops_mini"),
    mask_params=None,
    shuffle=False,
    seed=42,
    cache_dir="./cache_test_mini"
)

: 

: 

In [ ]:
dl = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
)

In [ ]:
features_list = []
labels = []
for entry in tqdm(dl):
    with torch.no_grad():
        x = entry["x"]
        mask = entry["mask"]
        encodings = (
            pretrained_model.encoder(
                x, mask=mask,
            )
            .cpu()
            .numpy()
        )
        features_list.append(encodings)
        labels.append(entry["y"].cpu().numpy())
features_np = np.concatenate(features_list)

100%|███████████████████████████████████████████████████████████████████████████████| 2500/2500 [30:27<00:00,  1.37it/s]


In [ ]:
all_labels = torch.cat([batch["y"] for batch in dl], dim=0)
labels = all_labels.cpu().numpy()

We use `features_np` to train a Random Forest classifier:

In [ ]:
model = RandomForestClassifier(class_weight="balanced", random_state=42)
model.fit(features_np, labels)

RandomForestClassifier(class_weight='balanced', random_state=42)

We can then use this trained random forest to make some predictions on the test data.

In [ ]:
dl_test = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
)

In [ ]:
test_preds = []
test_labels = []
for entry in tqdm(dl_test):
    with torch.no_grad():
        pretrained_model.eval()
        x = entry["x"]
        mask = entry["mask"]
        encodings = (pretrained_model.encoder(
            x, mask=mask)
            .cpu()
            .numpy()
        )
        test_preds.append(model.predict_proba(encodings))
        test_labels.append(entry["y"].cpu().numpy())

100%|█████████████████████████████████████████████████████████████████████████████████| 625/625 [07:44<00:00,  1.35it/s]


In [ ]:
test_labels_np = np.concatenate(test_labels)

In [ ]:
test_preds_np = np.concatenate(test_preds)
test_preds_np = np.argmax(test_preds_np, axis=1)

In [ ]:
f1 = f1_score(test_labels_np, test_preds_np, average="macro")
print(f"F1 score: {f1}")

F1 score: 0.9193620455921847
